# 08. Первое обучение своего Transformer и генерация

Лекция: [Обучение и генерация](../../notes/interview-prep/08l-first-training.md).

**Цель:** обновить параметры своей модели на тексте, сравнить train/validation loss и увидеть сгенерированное продолжение до и после обучения.

Данные — явно синтетический учебный корпус, создаваемый ниже из шаблона. Это не внешний датасет и не доказательство качества на обычном русском тексте. Интернет и новые библиотеки не нужны: Python + PyTorch.

Классы перенесены из твоего урока 07 без изменения реализации. Основное задание — три команды в train_step. A знакомит с данными, B запускает обучение, C показывает генерацию. Можно пройти всё за одно занятие; если данные непонятны, остановись после A.

Ментор не исполнял код, не проверял формат через валидатор и не рендерил ноутбук по договорённости. Outputs пока пустые. Первый запуск выполняешь ты в окружении deepschool; после заполнения задания: Restart Kernel → Run All → сохранить. Не делай Run All до заполнения трёх строк в B. Коммит отдельно по просьбе.

## Настройки запуска

Для первого запуска оставь значения ниже. Модель маленькая; используется CUDA при доступности, иначе CPU. Это стартовые настройки, не замеренная оптимальная конфигурация. Время зависит от твоего компьютера. Если долго, прерви ячейку и пришли вывод, не наращивай модель.

Seed помогает повторить опыт, но не гарантирует побитовую одинаковость между разными устройствами и версиями PyTorch.

In [1]:
import math
import random
import time
import torch
from torch import nn
from torch.nn import functional as F

SEED = 42
CONTEXT_LENGTH = 32
BATCH_SIZE = 16
TRAIN_STEPS = 300
EVAL_INTERVAL = 50
EVAL_BATCHES = 4
LEARNING_RATE = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
print('PyTorch:', torch.__version__)
print('Устройство:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


PyTorch: 2.12.1+cu126
Устройство: cuda
GPU: NVIDIA GeForce RTX 4070 Ti SUPER


## A. Данные: теперь один токен — один символ

В прошлом уроке токенами были слова из словаря из шести элементов. Теперь токены — буквы, пробел, точка и перевод строки. Архитектура не меняется: Embedding по-прежнему получает целочисленные ID.

Создаём 120 коротких историй (6 героев × 4 места × 5 действий). Делим целые истории 90/10 **до создания окон**. Шаблон у частей общий: validation оценивает новые комбинации знакомых конструкций, не независимый литературный корпус.

In [2]:
heroes = ['кот', 'пёс', 'ёж', 'кролик', 'барсук', 'лисенок']
places = ['у реки', 'в саду', 'у дома', 'на поляне']
actions = ['читал книгу', 'слушал птиц', 'искал грибы', 'смотрел на небо', 'собирал листья']
stories = [
    f'утром {hero} гулял {place}. потом он {action}. вечером {hero} вернулся домой.\n'
    for hero in heroes
    for place in places
    for action in actions
]
random.Random(SEED).shuffle(stories)
split_index = int(0.9 * len(stories))
train_stories = stories[:split_index]
val_stories = stories[split_index:]
train_text = ''.join(train_stories)
val_text = ''.join(val_stories)

# Алфавит задан заранее, а не извлечён из validation.
characters = list('\n .абвгдеёжзийклмнопрстуфхцчшщъыьэюя')
stoi = {char: i for i, char in enumerate(characters)}
itos = {i: char for char, i in stoi.items()}
vocab_size = len(characters)

def encode(text):
    unknown = set(text) - set(stoi)
    if unknown:
        raise ValueError(f'Нет в алфавите: {unknown!r}. Используй строчные русские буквы, пробел, точку и перенос строки.')
    return [stoi[char] for char in text]

def decode(ids):
    return ''.join(itos[int(i)] for i in ids)

train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)
print('Историй train / validation:', len(train_stories), len(val_stories))
print('Символов train / validation:', len(train_data), len(val_data))
print('Размер словаря:', vocab_size)
print('Пример истории:', train_stories[0])
print('ID строки кот:', encode('кот'))
print('Обратно:', decode(encode('кот')))


Историй train / validation: 108 12
Символов train / validation: 8549 949
Размер словаря: 36
Пример истории: утром кот гулял в саду. потом он собирал листья. вечером кот вернулся домой.

ID строки кот: [14, 18, 22]
Обратно: кот


### Сдвинутые окна

Берём отрезок длины T+1. Первые T символов — inputs, последние T — targets. Каждая позиция предсказывает следующий символ. Окна разных batch могут пересекаться внутри train — это нормально. Между train и validation общей исходной истории нет, но повторяющиеся фразы шаблона есть.

batch здесь — набор случайных окон, а не полный проход по данным. Один шаг не равен эпохе. Окна могут пересекать перевод строки между историями своей части.

In [3]:
def get_batch(data, batch_size=BATCH_SIZE, context_length=CONTEXT_LENGTH, generator=None):
    if len(data) <= context_length:
        raise ValueError('Нужно минимум context_length + 1 символов')
    starts = torch.randint(len(data) - context_length, (batch_size,), generator=generator)
    x = torch.stack([data[i:i + context_length] for i in starts.tolist()])
    y = torch.stack([data[i + 1:i + context_length + 1] for i in starts.tolist()])
    return x.to(DEVICE), y.to(DEVICE)

preview_rng = torch.Generator().manual_seed(123)
xb, yb = get_batch(train_data, generator=preview_rng)
print('Формы:', xb.shape, yb.shape)
print('Вход первой строки:', repr(decode(xb[0].tolist())))
print('Цель первой строки:', repr(decode(yb[0].tolist())))
print('Первые четыре пары:', [(itos[a], itos[b]) for a, b in zip(xb[0, :4].tolist(), yb[0, :4].tolist())])


Формы: torch.Size([16, 32]) torch.Size([16, 32])
Вход первой строки: ' читал книгу. вечером ёж вернулс'
Цель первой строки: 'читал книгу. вечером ёж вернулся'
Первые четыре пары: [(' ', 'ч'), ('ч', 'и'), ('и', 'т'), ('т', 'а')]


### Вопросы A

1. Если CONTEXT_LENGTH=32, сколько исходных символов нужно для одной пары input/target?
2. Что предсказываем после буквы «к» в начале слова «кот»: слово «кот» или символ «о»?

**Мои ответы:**

1. 33 символа
2. Символ о

Если с окнами понятно — продолжай к B. Перед циклом обязательно заполни train_step.

## Готовая модель из урока 07

Три класса перенесены без изменения кода. Новую архитектуру здесь не изучаем; ниже создадим экземпляр с D=32, четырьмя головами и двумя блоками.

In [4]:
class VectorizedCausalAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model <= 0 or num_heads <= 0 or d_model % num_heads != 0:
            raise ValueError('Нужны положительные D,H и D, кратное H')
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)
        self.output_projection = nn.Linear(d_model, d_model, bias=False)

    def project_heads(self, x):
        if x.ndim != 3 or x.shape[-1] != self.d_model or x.shape[1] == 0:
            raise ValueError('Ожидается непустой x формы (B,T,D)')
        batch_size, length, _ = x.shape
        # Готовая реализация из урока 05c: проекция и разделение голов.
        q, k, v = [
            projection(x)
            .reshape(batch_size, length, self.num_heads, self.head_dim)
            .transpose(1, 2)
            for projection in (
                self.query_projection,
                self.key_projection,
                self.value_projection,
            )
        ]

        return q, k, v

    def forward(self, x):
        batch_size, length, _ = x.shape
        q, k, v = self.project_heads(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        positions = torch.arange(x.shape[1], device=x.device)
        blocked = positions[None, :] > positions[:, None]
        masked_scores = scores.masked_fill(blocked, float('-inf'))
        weights = torch.softmax(masked_scores, dim=-1)
        context = weights @ v
        combined = context.transpose(1, 2).reshape(batch_size, length, self.d_model)
        return self.output_projection(combined)

In [5]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.norm_1 = nn.LayerNorm(d_model)
        self.attention = VectorizedCausalAttention(d_model, num_heads)
        self.norm_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        x = x + self.attention(self.norm_1(x))
        x = x + self.mlp(self.norm_2(x))
        return x


In [6]:
class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size, max_seq_len=16, d_model=8,
                 num_heads=2, d_ff=32, n_layers=1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids):
        if token_ids.ndim != 2:
            raise ValueError('Ожидаются token IDs формы (B,T)')
        length = token_ids.shape[1]
        if not 1 <= length <= self.max_seq_len:
            raise ValueError('Длина должна быть от 1 до max_seq_len')
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits


## Служебные функции — готовый код

estimate_loss оценивает одни и те же небольшие наборы окон при каждом вызове: отдельные генераторы случайных чисел не меняют выбор будущих обучающих batch. Это выборочная оценка, не loss на всей части данных. Batch одинакового размера, padding нет, поэтому среднее batch-loss соответствует среднему по выбранным позициям.

Декоратор @torch.no_grad() отключает запись графа внутри функции. eval переключает режим модели. После функции прежний режим восстанавливается. Это не обновляет веса.

In [7]:
@torch.no_grad()
def estimate_loss(model):
    was_training = model.training
    model.eval()
    results = {}
    try:
        for split, data, seed in [('train', train_data, 1001), ('val', val_data, 1002)]:
            rng = torch.Generator().manual_seed(seed)
            losses = []
            for _ in range(EVAL_BATCHES):
                x, y = get_batch(data, generator=rng)
                logits = model(x)
                loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
                losses.append(loss.item())
            results[split] = sum(losses) / len(losses)
    finally:
        model.train(was_training)
    return results


### Генерация: дописываем по одному символу

generate уже готова. Каждый проход берёт последние CONTEXT_LENGTH символов, получает logits последней позиции, превращает их в вероятности, случайно выбирает следующий ID и дописывает его. Для воспроизводимого сравнения используем отдельный seed. Выбор на CPU позволяет использовать один CPU-генератор и при модели на CUDA.

temperature=1 оставляет распределение неизменным. Другие температуры подробно разберём позже. Здесь фиксируем число новых символов, EOS-токена нет.

In [8]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=160, temperature=1.0, seed=2026):
    if not prompt:
        raise ValueError('Нужен непустой prompt')
    if temperature <= 0:
        raise ValueError('temperature должна быть положительной')
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    sample_rng = torch.Generator().manual_seed(seed)
    was_training = model.training
    model.eval()
    try:
        for _ in range(max_new_tokens):
            context = ids[:, -model.max_seq_len:]
            logits = model(context)
            probabilities = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            next_id = torch.multinomial(probabilities.cpu(), num_samples=1, generator=sample_rng).to(DEVICE)
            ids = torch.cat([ids, next_id], dim=1)
    finally:
        model.train(was_training)
    return decode(ids[0].tolist())


## B. Создать модель и увидеть результат ДО обучения

Внимание: повтор этой ячейки создаёт новую модель со случайной инициализацией и новый optimizer, стирая прогресс обучения в переменной model. Чтобы продолжить обучение существующей модели, повторяй только ячейку цикла ниже.

Выбран Adam с lr=0.001. Подробности адаптивного обновления разберём отдельно; роли zero_grad/backward/step остаются прежними.

In [9]:
torch.manual_seed(SEED)
model = TinyLanguageModel(
    vocab_size=vocab_size,
    max_seq_len=CONTEXT_LENGTH,
    d_model=32,
    num_heads=4,
    d_ff=128,
    n_layers=2,
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_rng = torch.Generator().manual_seed(SEED + 1)
steps_done = 0
history = []
PROMPT = 'утром кот '

before_losses = estimate_loss(model)
before_text = generate(model, PROMPT)
history.append({'step': 0, **before_losses})
print('Параметров:', sum(p.numel() for p in model.parameters()))
print('До обучения:', before_losses)
print('Ориентир для равномерного распределения ln(V):', math.log(vocab_size))
print('Генерация до обучения:\n' + before_text)


Параметров: 28544
До обучения: {'train': 3.688386917114258, 'val': 3.6963712573051453}
Ориентир для равномерного распределения ln(V): 3.58351893845611
Генерация до обучения:
утром кот ёизыэоеяр.шдкгъкые
юбётжижь.шскчузсбдежжк.кёзбпсцйфцёйфьгриаыютджзаяджьзааэачабазяоаахрбцвявуыьйз
.бялзажжяфщшаофдцрр пгаынпрнщгжяюйьёуаажр щхяеёауы юедяиаёъюрь


### Твоё задание: три команды обучения

Заменяй каждую строку raise NotImplementedError нужным вызовом. Эти исключения намеренные: они не дадут незаметно запустить цикл без обновления весов.

Порядок:

1. Очистить старые градиенты: optimizer.zero_grad(set_to_none=True).
2. После вычисления loss вызвать backward.
3. После вычисления градиентов вызвать optimizer.step.

Внутри этой функции НЕ нужен no_grad. Готовый контроль конечности loss только остановит обучение при inf/nan; он не исправляет причину.

In [12]:
def train_step(model, optimizer, x, y):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
    if not torch.isfinite(loss).item():
        raise RuntimeError('Loss стал inf/nan: остановись и пришли вывод')
    loss.backward()
    optimizer.step()
    return loss.item()


### Запустить обучение

После заполнения train_step выполняй эту ячейку. Сначала будет лог шага 1, затем каждые 50 шагов и в конце. step_loss посчитан на обучающем batch ДО обновления, train/val — на фиксированных оценочных окнах ПОСЛЕ него. Не сравнивай их как одно и то же измерение.

300 — число шагов, не эпох и не гарантия хорошего текста. Повтор этой ячейки продолжает текущую модель ещё на TRAIN_STEPS шагов, не начинает с нуля.

In [13]:
started = time.perf_counter()
for local_step in range(1, TRAIN_STEPS + 1):
    xb, yb = get_batch(train_data, generator=train_rng)
    step_loss = train_step(model, optimizer, xb, yb)
    steps_done += 1
    if local_step == 1 or steps_done % EVAL_INTERVAL == 0 or local_step == TRAIN_STEPS:
        metrics = estimate_loss(model)
        history.append({'step': steps_done, **metrics})
        elapsed = time.perf_counter() - started
        print(f'step={steps_done:4d} | step_loss={step_loss:.3f} | '
              f'train={metrics["train"]:.3f} | val={metrics["val"]:.3f} | '
              f'elapsed={elapsed:.1f}s', flush=True)


step=   1 | step_loss=3.683 | train=3.649 | val=3.655 | elapsed=0.1s
step=  50 | step_loss=2.488 | train=2.482 | val=2.428 | elapsed=0.4s
step= 100 | step_loss=1.843 | train=1.895 | val=1.845 | elapsed=0.6s
step= 150 | step_loss=1.597 | train=1.556 | val=1.538 | elapsed=0.9s
step= 200 | step_loss=1.254 | train=1.257 | val=1.234 | elapsed=1.1s
step= 250 | step_loss=1.087 | train=1.017 | val=0.997 | elapsed=1.3s
step= 300 | step_loss=0.929 | train=0.823 | val=0.820 | elapsed=1.5s


## C. Генерация ПОСЛЕ обучения

Те же prompt, seed, температура и длина. Это делает пример сравнимее, но один текст не является метрикой качества. Ищем появление знакомых сочетаний букв и структуры; не требуем идеальных предложений.

In [14]:
after_losses = estimate_loss(model)
after_text = generate(model, PROMPT)
print('Всего шагов:', steps_done)
print('Loss до:', before_losses)
print('Loss после:', after_losses)
print('\nДО:\n' + before_text)
print('\nПОСЛЕ:\n' + after_text)


Всего шагов: 300
Loss до: {'train': 3.688386917114258, 'val': 3.6963712573051453}
Loss после: {'train': 0.8232935965061188, 'val': 0.8197048455476761}

ДО:
утром кот ёизыэоеяр.шдкгъкые
юбётжижь.шскчузсбдежжк.кёзбпсцйфцёйфьгриаыютджзаяджьзааэачабазяоаахрбцвявуыьйз
.бялзажжяфщшаофдцрр пгаынпрнщгжяюйьёуаажр щхяеёауы юедяиаёъюрь

ПОСЛЕ:
утром кот птон оел . н гребы. биречерикром вером трокрнук лся. нулире.
утром пдом ёжрой.
утром л к вевером вечеролгулся. домомой.
утром пгул неляна рел чера домеромома ро


### Итоговые вопросы — коротко, своими словами

1. Уменьшились ли train loss и validation loss? Запиши свои числа до/после, не ожидаемые значения.
2. Что изменилось в сгенерированном тексте? Если ничего — так и напиши.
3. Какая команда действительно изменила параметры? Что было бы без backward?
4. При генерации модель получает правильный следующий символ из targets или дописывает свой выбранный символ?

**Мои ответы:**

1. Да, уменьшился. Loss до: {'train': 3.688386917114258, 'val': 3.6963712573051453}
Loss после: {'train': 0.8232935965061188, 'val': 0.8197048455476761}
2. Он стал частично осмысленным
3. optimizer.step() изенила, без backward градиент был бы нулевым и веса не поменялись бы
4. Мы считаем софтмакс последнего симвала и по нему беререм символ

Сохрани outputs и пришли на разбор. Автоматического сохранения весов на диск здесь нет: модель остаётся в памяти kernel. Не перезапускай kernel после обучения, если хочешь продолжить этот запуск. Для воспроизведения с нуля после заполнения TODO: Restart Kernel → Run All.

Следующий урок: разберём твои реальные результаты и возможное переобучение. Этот синтетический корпус имеет сильные повторы, поэтому даже хороший validation loss не доказывает качество на обычном тексте.

**Разбор ментора (исходные ответы сохранены):**

По сохранённым outputs выполнено 300 шагов; модель имеет 28 544 параметра. Train loss: 3.68839 → 0.82329, validation loss: 3.69637 → 0.81970. Это оценки на фиксированных выборках окон синтетического корпуса, не независимая оценка владения русским языком. Код ментором не запускался.

1. Верно: оба loss уменьшились.
2. Точнее: появились знакомые сочетания букв и фрагменты («утром», «вечером» внутри составных фрагментов, «домой» внутри искажённых слов); связного смысла пока нет. Один образец не является метрикой качества.
3. optimizer.step() указан верно. После zero_grad(set_to_none=True) без backward градиенты в данном цикле останутся None, а не нулевыми тензорами; Adam пропустит такие параметры. Не обобщаем на нулевые тензоры градиентов: у оптимизатора может быть накопленное состояние.
4. Направление верное. Берём вектор logits последней позиции по всем 36 символам словаря; softmax превращает его в распределение. torch.multinomial выбирает ID по вероятностям (не обязательно максимум). Дописываем собственный выбранный символ, не target.

Ответы A верны: нужно 33 исходных символа, предсказываем символ «о». train_step реализован правильно. Служебные функции оценки и генерации пока разбираем отдельно; понимание всех их строк не считаем завершённым только по успешному запуску.
